# Investigate and setup paths

In [1]:
from pathlib import Path

In [2]:
print(Path.cwd())

/Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks


In [3]:
for path in sorted(Path.cwd().iterdir()): 
    print(" -", path)

 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/.DS_Store
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/.ipynb_checkpoints
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/data
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/data_collection.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/evals
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/evals.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/mlops_prep_llm_training_serving.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/cour

In [4]:
config_path = Path("../configs/config.yaml")

# Read the configs

In [5]:
import yaml

In [6]:
with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

In [7]:
print(type(config))

<class 'dict'>


In [8]:
print(config)

{'model': {'name': 'Qwen/Qwen2.5-1.5B'}, 'data': {'path': 'data/01_cold_start_cot_sft/data.jsonl'}, 'output': {'directory': 'models/adapter_experiment'}, 'peft': {'method': 'lora', 'r': 16, 'alpha': 32, 'dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}, 'training': {'num_train_epochs': 3, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'learning_rate': 0.1002, 'max_length': 1024, 'seed': 42}}


In [9]:
from pprint import pprint

In [10]:
print("Model config:")
pprint(config["model"], sort_dicts=False)

Model config:
{'name': 'Qwen/Qwen2.5-1.5B'}


## Reading configs in variables

In [11]:
model_name = config["model"]["name"]
data_path = config["data"]["path"]
peft_method = config["peft"]["method"]
learning_rate = config["training"]["learning_rate"]

In [12]:
print("Model:", model_name)
print("Data:", data_path)
print("PEFT method:", peft_method)
print("Learning rate:", learning_rate)

Model: Qwen/Qwen2.5-1.5B
Data: data/01_cold_start_cot_sft/data.jsonl
PEFT method: lora
Learning rate: 0.1002


In [13]:
values_to_check = {
    "model_name": model_name,
    "data_path": data_path,
    "peft_method": peft_method,
    "learning_rate": learning_rate,
    "target_modules": config["peft"]["target_modules"],
}

In [14]:
for name, value in values_to_check.items():
    print(f"{name}: {value!r} | type={type(value).__name__}")

model_name: 'Qwen/Qwen2.5-1.5B' | type=str
data_path: 'data/01_cold_start_cot_sft/data.jsonl' | type=str
peft_method: 'lora' | type=str
learning_rate: 0.1002 | type=float
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] | type=list


## Validation configuations 

In [15]:
assert isinstance(config, dict)
assert config["model"]["name"]
assert config["peft"]["method"] in {"lora", "adalora", "ia3"}
assert config["training"]["learning_rate"] > 0
assert isinstance(config["peft"]["target_modules"], list)


# Connecting the above config to the existing TrainConfig class

## data and path shenanigans

In [18]:
from dataclasses import asdict
from pathlib import Path
from llm_training.config import TrainConfig

In [17]:
import sys
sys.path.append('../src/')

In [19]:
project_root = config_path.parent.parent

In [20]:
print(project_root)

..


In [21]:
data_path = project_root / config["data"]["path"]
output_dir = project_root / config["output"]["directory"]

In [23]:
print(output_dir)

../models/adapter_experiment


In [24]:
print("Data path:", data_path)
print("Exists:", data_path.exists())
print("Output directory:", output_dir)

Data path: ../data/01_cold_start_cot_sft/data.jsonl
Exists: True
Output directory: ../models/adapter_experiment


## Building TrainConfig class instance

In [25]:
train_config = TrainConfig(
    model_name=config["model"]["name"],
    data_path=data_path,
    output_dir=output_dir,
    lora_r=config["peft"]["r"],
    lora_alpha=config["peft"]["alpha"],
    lora_dropout=config["peft"]["dropout"],
    lora_target_modules=tuple(config["peft"]["target_modules"]),
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    learning_rate=config["training"]["learning_rate"],
    max_length=config["training"]["max_length"],
    seed=config["training"]["seed"],
)

In [26]:
for name, value in asdict(train_config).items():
    print(f"{name}: {value!r}")

model_name: 'Qwen/Qwen2.5-1.5B'
data_path: PosixPath('../data/01_cold_start_cot_sft/data.jsonl')
output_dir: PosixPath('../models/adapter_experiment')
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target_modules: ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
num_train_epochs: 3
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 0.1002
max_length: 1024
warmup_ratio: 0.03
lr_scheduler_type: 'cosine'
logging_steps: 10
eval_steps: 50
save_steps: 200
save_total_limit: 2
max_steps: None
fp16: False
bf16: True
seed: 42
report_to: 'none'


# Connecting the config to messages now 

In [27]:
from llm_training.data.messages import (
    load_jsonl,
    to_message,
    to_sft_dataset,
)

## Loadigng and inspecting records

In [28]:
records = load_jsonl(train_config.data_path)

In [30]:
print(records[0])

{'spec': {'topic': 'damaged-item', 'tone': 'confused', 'item': 'running sneakers', 'days_since_purchase': 6, 'payment_method': 'gift card', 'order_id': 'ORD-201414'}, 'prompt': "Hi, I just opened my order of running sneakers (ORD-201414) and there's a big tear in the side. I'm not sure what to do – can I return them? I paid with a gift card and I'm worried I'll lose the money.", 'response': "<thinking>The customer is reporting a damaged item (running sneakers) from order ORD-201414, which was purchased 6 days ago. According to policy, the standard return window is 30 days from delivery date, and this order is within that window. Additionally, damaged or wrong items are always refundable with a free return label, even outside the 30-day window. Since the item is damaged, we can authorize a return. The original payment method was a gift card, so the refund will go back to the same gift card within 5-7 business days after the return is processed. The customer is confused and worried about

In [32]:
assert "prompt" in records[10]
assert "response" in records[10]

## Converting one record (passing through our messages)

In [34]:
message_record = to_message(records[0])

In [36]:
message_record

{'messages': [{'role': 'user',
   'content': "Hi, I just opened my order of running sneakers (ORD-201414) and there's a big tear in the side. I'm not sure what to do – can I return them? I paid with a gift card and I'm worried I'll lose the money."},
  {'role': 'assistant',
   'content': "<thinking>The customer is reporting a damaged item (running sneakers) from order ORD-201414, which was purchased 6 days ago. According to policy, the standard return window is 30 days from delivery date, and this order is within that window. Additionally, damaged or wrong items are always refundable with a free return label, even outside the 30-day window. Since the item is damaged, we can authorize a return. The original payment method was a gift card, so the refund will go back to the same gift card within 5-7 business days after the return is processed. The customer is confused and worried about losing the money, so we need to reassure them and explain the process clearly. We'll provide a free retu

## Creating the HF dataset

In [39]:
train_ds, eval_ds = to_sft_dataset(
    train_config.data_path,
    val_frac=0.1,
    seed=train_config.seed,
)

In [40]:
print("Train rows:", len(train_ds))
print("Eval rows:", len(eval_ds))
print("Columns:", train_ds.column_names)


Train rows: 446
Eval rows: 50
Columns: ['spec', 'prompt', 'response', 'messages']


In [41]:
train_ds[0]

{'spec': {'topic': 'exchange-size',
  'tone': 'urgent',
  'item': 'kids pajamas',
  'days_since_purchase': 41,
  'payment_method': 'gift card',
  'order_id': 'ORD-667296'},
 'prompt': "I need to exchange the kids pajamas I bought from order ORD-667296 for a different size. They are too small. I bought them 41 days ago and I'm in a hurry because my child needs them for a school event. Can you help me with an exchange or refund?",
 'response': "<thinking>Customer is requesting an exchange for a different size due to fit issue. Item was purchased 41 days ago, which is beyond the 30-day return window from delivery date. Payment was made via gift card on order ORD-667296. Policy states that standard returns/exchanges are only accepted within 30 days of delivery. The item is not damaged or wrong (size issue is not covered under the exception for damaged or wrong items). Therefore, we cannot process a return or exchange under the current policy. Since the customer is urgent, we should apologi

## Inspecting model settings

In [42]:
model_name = config["model"]["name"]

In [43]:
model_name

'Qwen/Qwen2.5-1.5B'

In [44]:
peft_config = config["peft"]

In [45]:
peft_config

{'method': 'lora',
 'r': 16,
 'alpha': 32,
 'dropout': 0.05,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj']}

## Build LoRA configuration

In [48]:
from peft import LoraConfig
if peft_config["method"] != "lora":
    raise NotImplementedError(
        f"PEFT method not implemented yet: {peft_config['method']}"
    )

In [49]:
lora_config = LoraConfig(
    r=peft_config["r"],
    lora_alpha=peft_config["alpha"],
    lora_dropout=peft_config["dropout"],
    target_modules=peft_config["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

## Inspect the generated PEFT Obect

In [51]:
print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'down_proj', 'q_proj', 'v_proj', 'gate_proj', 'k_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [52]:
assert lora_config.r == config["peft"]["r"]
assert lora_config.lora_alpha == config["peft"]["alpha"]
assert lora_config.lora_dropout == config["peft"]["dropout"]
assert lora_config.target_modules == set(config["peft"]["target_modules"])


## Testing invalid method handling

In [53]:
test_method = "ia3"
if test_method != "lora":
    print(f"Would require a different PEFT config: {test_method}")

Would require a different PEFT config: ia3


### The proves that different PEFT algorithms do not silently receive LoRA-specific parameters